# Milestones 1, 2 & 3 — Google Colab Benchmark & Full Pipeline Runner (CPU-Only)

**Cấu trúc theo dõi từng giai đoạn trong Notebook này:**
- **Bước 1 $\rightarrow$ Bước 4**: Chuẩn bị môi trường, kết nối Google Drive, kiểm tra phần cứng CPU-only và chọn chế độ chạy (`PILOT` / `FULL_MEMBER`).
- **Bước 5 $\rightarrow$ Bước 6 (Milestone 1 — Data Crawling)**: Chạy `Discovery` $\rightarrow$ `Download` $\rightarrow$ `Extract` $\rightarrow$ `References` $\rightarrow$ `Export (papers, references)` và xem bảng thống kê hiệu năng M1.
- **Bước 7 (Milestone 2 — Hierarchical Parsing & Standardization)**: Phân tích phân cấp LaTeX, chuẩn hóa toán, khử trùng lặp phần tử & BibTeX, xuất 3 bảng Parquet (`elements`, `hierarchy`, `bibtex`) và in thống kê hiệu năng M2.
- **Bước 8 (Milestone 3 — Reference Matching Pipeline)**: Chạy gán nhãn tự động ($\ge 10\%$), tích hợp nhãn thủ công, huấn luyện mô hình Logistic Candidate Ranker (8 đặc trưng), cập nhật bảng `predictions` và in kết quả đánh giá $\text{MRR@5}$.
- **Bước 9 (Pre-submission Validation & Upload)**: Kiểm định toàn bộ **6 bảng Parquet** bằng `scripts/validate_hf_dataset.py`, đẩy lên **Hugging Face** và tải các file JSON báo cáo về máy.

## Bước 1: Chuẩn bị môi trường & Cấu hình khóa API trên Google Colab

- Đảm bảo trên Google Colab đã chọn **Runtime $\rightarrow$ Change runtime type $\rightarrow$ Hardware accelerator: None (CPU)**.
- Nếu có `SEMANTIC_SCHOLAR_API_KEY` hoặc `HF_TOKEN`, bạn có thể lưu trong mục **Secrets (biểu tượng chìa khóa 🔑 bên thanh trái của Colab)**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/theanhnek/arxiv-data-project.git"
REPO_BRANCH = "main"

if IS_COLAB:
    print("[Colab] Đang thiết lập môi trường trên Google Colab...")
    repo_dir = Path("/content/arxiv-data-project")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", REPO_BRANCH], check=False)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)

    try:
        from google.colab import userdata
        for secret_name in ("ARXIV_USER_AGENT", "SEMANTIC_SCHOLAR_API_KEY", "HF_TOKEN"):
            try:
                val = userdata.get(secret_name)
                if val and not os.getenv(secret_name):
                    os.environ[secret_name] = val.strip()
                    print(f"[Colab Secrets] Đã nạp {secret_name}")
            except Exception:
                pass
    except ImportError:
        pass
else:
    if Path.cwd().name == "src":
        os.chdir(Path.cwd().parent)
    print(f"[Local] Đang chạy trên máy cá nhân tại thư mục: {Path.cwd()}")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

os.environ.setdefault("ARXIV_USER_AGENT", "Lab1ArxivCrawler/0.1 (HCMUS Intro-DS Group 14)")
print("Thư mục làm việc hiện tại:", Path.cwd())
print("Chế độ Semantic Scholar API Key:", "Đã cấu hình (1.0 req/s)" if os.getenv("SEMANTIC_SCHOLAR_API_KEY") else "Không có key (3.1 s/req)")

## Bước 2: (Khuyên dùng khi chạy Full 5.000 bài) Kết nối Google Drive để lưu tiến độ

In [ ]:
USE_GOOGLE_DRIVE = False  # Đổi thành True nếu muốn lưu thẳng vào Google Drive trên Colab

if IS_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/arxiv-data-project/data")
else:
    OUTPUT_DIR = Path("data")

for sub in ("corpus", "state", "metrics", "tmp", "export", "parsed", "matching"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"Thư mục lưu dữ liệu: {OUTPUT_DIR.resolve()}")

## Bước 3: Kiểm tra cấu hình phần cứng (CPU-only Testbed Audit)

In [ ]:
import json
from src.metrics import collect_environment_info

env_info = collect_environment_info(OUTPUT_DIR)
if os.environ.get("COLAB_GPU"):
    print("Runtime đang bật GPU. Đề bài yêu cầu benchmark trên Google Colab chế độ CPU-only!")
else:
    print("Runtime đang ở chế độ CPU-only.")

print("=== THÔNG SỐ MÔI TRƯỜNG CHẠY BENCHMARK ===")
print(json.dumps(env_info, indent=2, ensure_ascii=False))

## Bước 4: Chọn chế độ chạy (`PILOT` hoặc `FULL_MEMBER`) và Thành viên (`1..5`)

In [ ]:
from src.discovery import prepare_assigned_ids

ASSIGNED_RANGES = {
    1: ("2025-10", 14401, "2025-10", 19400),  # Thành viên 1: Kiệt
    2: ("2025-10", 9401,  "2025-10", 14400),  # Thành viên 2: Dương
    3: ("2025-10", 4401,  "2025-10", 9400),   # Thành viên 3: Hà
    4: ("2025-09", 26047, "2025-10", 4400),   # Thành viên 4: Kiên
    5: ("2025-09", 11047, "2025-09", 16046),  # Thành viên 5: Anh
}

RUN_MODE = "PILOT"                  # Đổi thành "FULL_MEMBER" khi chạy thật toàn bộ 5.000 ID
MEMBER_INDEX = 2                    # Số thứ tự thành viên (1..5)
PILOT_COUNT = 12                    # Số bài báo chạy thử khi ở chế độ PILOT (>= 10 để chạy thử cả M3 automatic matching)
RESUME = True                       # True: tiếp tục từ kết quả đã chạy trước đó
CLEANUP_TMP_AFTER_EXTRACT = False   # Đổi thành True nếu muốn xóa file nén .tar.gz sau khi giải nén

start_m, start_id, end_m, end_id = ASSIGNED_RANGES[MEMBER_INDEX]
all_member_ids = prepare_assigned_ids(
    start_m,
    start_id,
    end_m,
    end_id,
    OUTPUT_DIR / "assigned_ids.txt",
    expected_total=5000,
)

if RUN_MODE == "PILOT":
    target_ids = all_member_ids[:PILOT_COUNT]
    print(f"[Chế độ PILOT] Chọn {len(target_ids)} bài đầu tiên của Thành viên {MEMBER_INDEX}: {target_ids[0]} -> {target_ids[-1]}")
else:
    target_ids = all_member_ids
    print(f"[Chế độ FULL_MEMBER] Chọn toàn bộ {len(target_ids)} bài của Thành viên {MEMBER_INDEX}: {target_ids[0]} -> {target_ids[-1]}")

---
## Bước 5: [Milestone 1] Chạy Data Crawling End-to-End và Đo Hiệu năng (Runtime, RAM, Disk)

Thực hiện tuần tự: **Discovery $\rightarrow$ Download $\rightarrow$ Extraction (.tex/.bib) $\rightarrow$ References (Semantic Scholar) $\rightarrow$ Export (`papers`, `references`)**.

In [ ]:
import os
import shutil
from src.config import CrawlConfig
from src.discovery import ArxivClient, discover_papers, write_json
from src.downloader import download_discovered_sources
from src.extractor import extract_downloaded_sources
from src.references import collect_references
from src.storage import export_parquet, export_tables
from src.metrics import ResourceMonitor, compute_pipeline_metrics

s2_api_key = (os.getenv("SEMANTIC_SCHOLAR_API_KEY") or "").strip() or None
config = CrawlConfig(
    output_dir=OUTPUT_DIR,
    arxiv_delay_seconds=3.0,
    request_timeout_seconds=60.0,
    max_attempts=3,
    user_agent=os.getenv("ARXIV_USER_AGENT", "Lab1ArxivCrawler/0.1 (HCMUS Intro-DS Group 14)"),
    s2_api_key=s2_api_key,
    s2_delay_seconds=1.0 if s2_api_key else 3.1,
    metrics_sample_interval_seconds=0.5,
)

write_json(
    OUTPUT_DIR / "state" / "input_validation.json",
    {"ids": target_ids, "invalid": [], "duplicates": []},
)

client = ArxivClient(config)
with ResourceMonitor(config.output_dir, config.metrics_sample_interval_seconds) as monitor:
    try:
        print("--- [1/6] Bắt đầu Discovery (Identify papers & versions) ---")
        disc_report = discover_papers(target_ids, config, resume=RESUME, client=client)

        print("\n--- [2/6] Bắt đầu Download (Fetch versioned sources) ---")
        dl_report = download_discovered_sources(config, resume=RESUME, client=client)

        print("\n--- [3/6] Bắt đầu Extraction (.tex / .bib) ---")
        ext_report = extract_downloaded_sources(config, resume=RESUME)

        print("\n--- [4/6] Bắt đầu References (Semantic Scholar) ---")
        ref_report = collect_references(config, resume=RESUME)

        print("\n--- [5/6] Xuất bảng dữ liệu Milestone 1 (JSONL & Sharded Parquet) ---")
        export_report = export_tables(config)
        parquet_report = export_parquet(config)
    finally:
        client.close()

if CLEANUP_TMP_AFTER_EXTRACT:
    tmp_dl_dir = OUTPUT_DIR / "tmp" / "downloads"
    if tmp_dl_dir.exists():
        shutil.rmtree(tmp_dl_dir, ignore_errors=True)
        print("Đã dọn dẹp thư mục file nén tạm data/tmp/downloads/ sau khi giải nén.")

print("\n--- [6/6] Tổng hợp Metrics Milestone 1 ---")
summary_metrics = compute_pipeline_metrics(config, monitor_snapshot=monitor.snapshot)
print("\n=== HOÀN TẤT MILESTONE 1! ĐÃ LƯU BÁO CÁO TẠI data/metrics/benchmark_summary.json ===")
print(f"Xuất bảng M1 thành công: {export_report['papers_rows']} bài và {export_report['references_rows']} trích dẫn.")

## Bước 6: [Milestone 1] Xem Bảng tổng kết Kết quả & Hiệu năng Crawling

In [ ]:
disc = summary_metrics["discovery"]
dlext = summary_metrics["download_and_extraction"]
sizes = summary_metrics["sizes_bytes"]
perf = summary_metrics["performance"]
res = perf["resources"]
refs = summary_metrics.get("references") or {}

print("=== TỔNG KẾT MILESTONE 1 (DATA CRAWLING) ===")
print(f"1. Số bài báo kiểm tra (Assigned IDs)        : {disc['assigned_unique_ids']}")
print(f"2. Số bài báo tìm thấy (Resolved Papers)     : {disc['resolved_papers']} ({disc['paper_discovery_success_rate']*100:.2f}%)")
print(f"3. Tổng số phiên bản (Discovered Versions)   : {disc['discovered_versions']}")
print(f"4. Số phiên bản tải & giải nén thành công    : {dlext['extracted_tex_versions']} có TeX + {dlext['no_tex_source_versions']} không có TeX ({dlext['version_completion_rate']*100:.2f}%)")
print(f"5. Kích thước TB 1 bài TRƯỚC khi bỏ ảnh      : {sizes['avg_uncompressed_bytes_per_paper_before_filtering']:,.0f} bytes")
print(f"6. Kích thước TB 1 bài SAU khi bỏ ảnh (.tex) : {sizes['avg_retained_bytes_per_paper_after_filtering']:,.0f} bytes")
print(f"7. Tỷ lệ tiết kiệm dung lượng nhờ bỏ ảnh     : {sizes['figure_removal_savings_ratio']*100:.2f}%")
print("-" * 65)
if refs and refs.get("available"):
    print(f"8. Số bài query References Semantic Scholar  : {refs['queried_papers']} ({refs['reference_scrape_success_rate']*100:.2f}%)")
    print(f"9. Tổng số trích dẫn thu thập được           : {refs['total_references_crawled']} (TB {refs['avg_references_per_queried_paper']:.1f} refs/bài)")
    print(f"10. Số trích dẫn chứa arXiv ID hợp lệ        : {refs['total_arxiv_references_crawled']}")
    print("-" * 65)
print(f"11. Thời gian chạy Discovery                 : {perf['discovery_elapsed_seconds']:.2f} giây")
print(f"12. Tổng thời gian chạy toàn tuyến (Wall time): {perf['end_to_end_wall_time_seconds']:.2f} giây")
print(f"13. Thời gian xử lý trung bình 1 bài báo     : {perf['avg_time_per_resolved_paper_seconds']:.2f} giây/bài")
print(f"14. Bộ nhớ RAM trung bình / cực đại          : {res['avg_ram_mb']:.2f} MB / {res['peak_ram_mb']:.2f} MB")
print(f"15. Dung lượng ổ đĩa đỉnh (Peak Disk)        : {res['peak_working_disk_mb']:.3f} MB")
print(f"16. Dung lượng Corpus cuối cùng (Final Disk) : {res['final_corpus_disk_mb']:.3f} MB")

---
## Bước 7: [Milestone 2] Chạy Hierarchical Parsing & Standardization (`elements`, `hierarchy`, `bibtex`)

- Tìm file gốc `\documentclass`, nối đệ quy `\input`/`\include`, dựng cây phân cấp (`document` $\rightarrow$ `heading` $\rightarrow$ `paragraph`/`list`/`list_item` $\rightarrow$ `sentence`/`formula`/`figure`).
- Chuẩn hóa toán inline `$...$` và block `\begin{equation}...\end{equation}`, loại bỏ mục References khỏi cây phân cấp.
- Khử trùng lặp phần tử văn bản và mục BibTeX xuyên phiên bản (kèm đổi tên `\cite{}`), xuất 3 bảng Parquet `elements`, `hierarchy`, `bibtex` và ghi báo cáo hiệu năng vào `data/parsed/milestone2_summary.json`.

In [ ]:
import json
from scripts.build_milestone2_tables import main as run_milestone2

print("=== BẮT ĐẦU MILESTONE 2: HIERARCHICAL PARSING & STANDARDIZATION ===")
run_milestone2(["--output", str(OUTPUT_DIR)])

m2_summary_path = OUTPUT_DIR / "parsed" / "milestone2_summary.json"
if m2_summary_path.is_file():
    m2 = json.loads(m2_summary_path.read_text(encoding="utf-8"))
    m2_res = m2.get("resources") or {}
    print("\n=== TỔNG KẾT MILESTONE 2 (PARSING & STANDARDIZATION) ===")
    print(f"1. Số bài báo xử lý (Resolved Papers)        : {m2['resolved_papers']}")
    print(f"2. Tổng số version kiểm tra                  : {m2['versions_seen']}")
    print(f"3. Số version parse thành công / rỗng        : {m2['versions_parsed']} parsed / {m2['versions_empty_or_no_root']} empty")
    print(f"4. Số dòng bảng `elements` (đã deduplicate)  : {m2['elements_rows']:,}")
    print(f"5. Số dòng bảng `hierarchy` (edges)          : {m2['hierarchy_rows']:,}")
    print(f"6. Số dòng bảng `bibtex` (đã hợp nhất)       : {m2['bibtex_rows']:,}")
    print(f"7. Thời gian chạy Parser (Wall time)         : {m2.get('elapsed_seconds', 0):.2f} giây ({m2.get('avg_seconds_per_paper', 0):.3f} giây/bài)")
    if m2_res:
        print(f"8. Bộ nhớ RAM trung bình / cực đại (Parser)  : {m2_res.get('avg_ram_mb', 0):.2f} MB / {m2_res.get('peak_ram_mb', 0):.2f} MB")
        print(f"9. Tổng dung lượng thư mục Output sau M2     : {m2_res.get('final_output_disk_mb', 0):.3f} MB")

---
## Bước 8: [Milestone 3] Chạy Reference Matching Pipeline (`predictions` & Đánh giá $\text{MRR@5}$)

Thực hiện tuần tự:
1. **Automatic Matching**: Gán nhãn heuristic ($0.85 \cdot \text{title\_dice} + 0.15 \cdot \text{author\_dice}$) trên $\ge 10\%$ số bài ngoài tập thủ công (tối thiểu 10 bài).
2. **Manual Label Integration**: Nếu đã có file `data/matching/manual_label_review.csv` với các dòng `CONFIRMED_BY_USER`, ghép các bài thủ công vào bảng `predictions` và cập nhật `data/matching/manual_labels.jsonl`.
3. **Supervised Model Training & Ranking**: Huấn luyện mô hình **Logistic Candidate Ranker** trên 8 đặc trưng (`title_dice`, `title_jaccard`, `bigram_dice`, `author_dice`, `title_exact`, `length_ratio`, `year_similarity`, `baseline_rank`), chấm điểm trên không gian ứng viên $m \times n$ của từng bài báo, đánh giá $\text{MRR@5}$ và cập nhật bảng `predictions`.

In [ ]:
import json
from scripts.build_automatic_predictions import main as run_auto_matching
from scripts.build_manual_predictions import main as run_manual_matching
from scripts.train_matcher import main as run_train_matcher
from scripts.promote_model_predictions import main as run_promote_predictions

print("--- [1/3] Gán nhãn tự động (Automatic Heuristic Matching) ---")
run_auto_matching(["--output", str(OUTPUT_DIR)])

review_csv = OUTPUT_DIR / "matching" / "manual_label_review.csv"
if review_csv.is_file():
    print("\n--- [2/3] Tích hợp nhãn thủ công đã xác nhận (Manual Labels) ---")
    try:
        run_manual_matching(["--output", str(OUTPUT_DIR)])
    except SystemExit as exc:
        print(f"[Lưu ý nhãn thủ công]: {exc}")
else:
    print("\n--- [2/3] Chưa có file manual_label_review.csv (bỏ qua bước ghép nhãn thủ công) ---")

print("\n--- [3/3] Huấn luyện Logistic Candidate Ranker, tính MRR@5 & Cập nhật bảng predictions ---")
try:
    run_train_matcher(["--output", str(OUTPUT_DIR)])
    run_promote_predictions(["--output", str(OUTPUT_DIR)])
except SystemExit as exc:
    print(f"[Lưu ý huấn luyện mô hình]: {exc}")

mrr_path = OUTPUT_DIR / "matching" / "mrr_evaluation.json"
if mrr_path.is_file():
    m3 = json.loads(mrr_path.read_text(encoding="utf-8"))
    prim = m3.get("primary_manual_evaluation") or {}
    all_eval = m3.get("test_all_evaluation") or {}
    print("\n=== TỔNG KẾT MILESTONE 3 (REFERENCE MATCHING & MRR@5) ===")
    print(f"1. Mô hình xếp hạng                : {m3.get('model')}")
    print(f"2. Các đặc trưng (8 features)      : {', '.join(m3.get('features', []))}")
    print(f"3. Số cặp huấn luyện (Pos / Total) : {m3.get('training_positive_pairs', 0):,} / {m3.get('training_pairs', 0):,}")
    print(f"4. Số dòng predictions theo tập    : {m3.get('partition_rows')}")
    print(f"5. Validation MRR@5 (Primary)      : {prim.get('valid_mrr_at_5', 0):.4f}")
    print(f"6. Test MRR@5 (Baseline -> Model)  : {prim.get('baseline_test_mrr_at_5', 0):.4f} -> {prim.get('test_mrr_at_5', 0):.4f}")
    if all_eval:
        print(f"7. Test MRR@5 (All test rows)      : {all_eval.get('baseline_test_mrr_at_5', 0):.4f} -> {all_eval.get('test_mrr_at_5', 0):.4f}")

---
## Bước 9: Kiểm định toàn bộ 6 bảng Parquet (`validate_hf_dataset.py`) & Upload lên Hugging Face

In [ ]:
from scripts.validate_hf_dataset import validate_dataset, _print_human_report
from src.hf_upload import upload_dataset

validation_report = validate_dataset(
    OUTPUT_DIR / "export" / "parquet",
    member=str(MEMBER_INDEX),
)
_print_human_report(validation_report)

UPLOAD_TO_HF = False  # Đổi thành True khi muốn đẩy bộ Parquet lên Hugging Face
HF_REPO_ID = "your-username/arxiv-g14-member2"  # Đổi thành <username>/<dataset-name> của bạn

if UPLOAD_TO_HF and validation_report["passed"]:
    hf_token = (os.getenv("HF_TOKEN") or "").strip() or None
    dataset_url = upload_dataset(HF_REPO_ID, OUTPUT_DIR / "export" / "parquet", token=hf_token)
    print("✅ Đã upload lên Hugging Face:", dataset_url)

if IS_COLAB:
    from google.colab import files
    for rel_path in (
        Path("metrics/benchmark_summary.json"),
        Path("metrics/per_paper_metrics.json"),
        Path("parsed/milestone2_summary.json"),
        Path("matching/mrr_evaluation.json"),
    ):
        fpath = OUTPUT_DIR / rel_path
        if fpath.is_file():
            files.download(str(fpath))